<a href="https://colab.research.google.com/github/Auta01/Tensorflow-cases/blob/main/Transfer%20learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Transfer learning with Tensorflow

In [23]:
##Download the and load the data
import zipfile

!wget https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip

#unzip file
zip_ref = zipfile.ZipFile('10_food_classes_10_percent.zip')
zip_ref.extractall()
zip_ref.close()


--2026-01-31 16:12:01--  https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.170.207, 173.194.174.207, 74.125.23.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.170.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 168546183 (161M) [application/zip]
Saving to: ‘10_food_classes_10_percent.zip.2’

10_food_classes_10_ 100%[===================>] 160.74M  32.0MB/s    in 6.0s    

2026-01-31 16:12:07 (26.7 MB/s) - ‘10_food_classes_10_percent.zip.2’ saved [168546183/168546183]



In [24]:
#How many images in each folder

import os

for dirpath, dirnames, filenames, in os.walk('10_classes_10_percent'):

  print(f'There are {len(dirnames)} directories and {len(filenames)} images in {len(dirpath)}''.')

In [25]:
##Creating data loader preparing

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224, 224)
BATCH_SIZE = 32

train_dir = '10_food_classes_10_percent/train/'
test_dir = '10_food_classes_10_percent/test/'

# Instantiate ImageDataGenerator for training and testing
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

print('Training images:')

train_data_10_percent = train_datagen.flow_from_directory(train_dir,
                                                          target_size=IMAGE_SHAPE,
                                                          batch_size=BATCH_SIZE,
                                                          class_mode='categorical')
print('Testing images:')

test_data = test_datagen.flow_from_directory(test_dir,
                                             target_size=IMAGE_SHAPE,
                                             batch_size=BATCH_SIZE,
                                             class_mode='categorical')

Training images:
Found 750 images belonging to 10 classes.
Testing images:
Found 2500 images belonging to 10 classes.


CallBacks

In [26]:
#Setting callback
#some popular callback
#1. Tracking experiment with the tensorflow callback

#2.Model checkpoint with the modelcheckpoint callback

#3.  Early-stopping a model from training(before it train too loong and overfit with the earlystopping callback)

#Create a tensorflow callback
import datetime

def create_tensorboard_callback(dir_name, experiment_name):

  log_dir = dir_name+ '/' + experiment_name +'/'+datetime.datetime.now().strftime('%Y%m%d,%H%M%S')
  tensorboard_callback = tf.keras.callback.TensorBoard(log_dir=log_dir)

  print(f'Saving TensorBoard log files to:' (log_dir))

  return tensorboard_callback



<>:17: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
<>:17: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
/tmp/ipython-input-3140377728.py:17: SyntaxWarning: 'str' object is not callable; perhaps you missed a comma?
  print(f'Saving TensorBoard log files to:' (log_dir))


In [27]:
#Creating model using tensorflow hub
resnet_url = 'https://tfhub.dev/google/imagenet/resnet_v2_50/feature_vector/4'

efficientnet_url = 'https://tfhub.dev/tensorflow/efficientnet/b0/feature-vector/1'

In [49]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers


In [55]:
#lets make a create model() function to create a model from aurl
def create_model(model_url,  num_classes=10):

  # Create the KerasLayer from the TF Hub URL
  feature_extractor_layer = hub.KerasLayer(model_url,
                                           trainable=False,
                                           name='feature_extraction_layer')

  # Create our model using the Functional API
  inputs = tf.keras.Input(shape=IMAGE_SHAPE + (3,))
  x = feature_extractor_layer(inputs)
  outputs = layers.Dense(num_classes, activation='softmax', name='output_layer')(x)
  model = tf.keras.Model(inputs=inputs, outputs=outputs)

  return model

In [59]:
resnet_model = create_model(resnet_url, num_classes=train_data_10_percent.num_classes)

# Compile the model
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Print the summary
resnet_model.summary()

ValueError: Only instances of `keras.Layer` can be added to a Sequential model. Received: <tensorflow_hub.keras_layer.KerasLayer object at 0x7a58f80b8d40> (of type <class 'tensorflow_hub.keras_layer.KerasLayer'>)

In [52]:
model.summary()

NameError: name 'model' is not defined